# Form initial table of schools

In [1]:
import pandas as pd
from src.changing_koatuu_schools_dict import *
from src.merging_with_id import *

In [2]:
# load datasets
datasets = {}
for year in range(2016, 2026):
    print(f"EIE {year} Loading...")
    if int(year) >= 2019:
        file_name = f'Odata{year}File.csv'
    else:
        file_name = f'OpenData{year}.csv'
    try:           
        dataset = pd.read_csv(f"../../data_loader/{year}/{file_name}", sep=";", encoding='utf-8', dtype = str)
    except UnicodeDecodeError :
        dataset = pd.read_csv(f"../../data_loader/{year}/{file_name}", sep=";", encoding='Windows 1251', dtype = str)
    datasets.update({year: dataset})
    print("success")

EIE 2016 Loading...
success
EIE 2017 Loading...
success
EIE 2018 Loading...
success
EIE 2019 Loading...
success
EIE 2020 Loading...
success
EIE 2021 Loading...
success
EIE 2022 Loading...
success
EIE 2023 Loading...
success
EIE 2024 Loading...
success
EIE 2025 Loading...
success


In [3]:
for year, dataset in datasets.items():
    dataset.columns = [col.lower() for col in dataset.columns]
    dataset['year'] = year

In [4]:
for year, dataset in datasets.items():
    print(year,":",[col for col in dataset.columns if col.startswith('eo')])

2016 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2017 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2018 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2019 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2020 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2021 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2022 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2023 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2024 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent']
2025 : ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoparent', 'eoedrpou', 'eoedeboid']


In [5]:
schools_edrpou = pd.DataFrame()

desired_cols = ['eoname', 'eotypename', 'eoregname', 'eoareaname', 'eotername', 'eoedrpou', 'year']

for year, dataset in datasets.items():
    temp_df = dataset.reindex(columns=desired_cols)
    schools_edrpou = pd.concat([schools_edrpou, temp_df], ignore_index=True)

In [6]:
schools_edrpou.drop_duplicates(inplace=True)
# When the paricipant graduated in the previous years 
# the corresponding information about school (everything with 'eo') is NaN.
# We drop these rows.
schools_edrpou.dropna(subset=schools_edrpou.columns.difference(['year']), 
                      how = 'all', inplace=True)
schools_edrpou.reset_index(drop=True, inplace=True)
schools_edrpou

,eoname,eotypename,eoregname,eoareaname,eotername,eoedrpou,year
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,Запорізька область,Мелітопольський район,с.Терпіння,NaN,2016
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,Хмельницька область,Красилівський район,м.Красилів,NaN,2016
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,Чернівецька область,м.Чернівці,Шевченківський район міста,NaN,2016
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,Донецька область,Донецька область,м.Дружківка,NaN,2016
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,Тернопільська область,Тернопільська область,м.Тернопіль,NaN,2016
...,...,...,...,...,...,...,...
99911,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,Львівська область,Стрийський район,м.Ходорів,05393837,2025
99912,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,м.Київ,м.Київ,Дарницький район міста,42372041,2025
99913,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,Одеська область,Роздільнянський район,с.Гаївка,25038104,2025
99914,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,Житомирська область,Житомирський район,с.Вільха,45498834,2025


In [7]:
schools_edrpou.year.value_counts()

year
2018    11001
2019    10876
2020    10743
2016    10492
2021    10467
2017    10081
2022     9752
2023     9204
2024     8828
2025     8472
Name: count, dtype: int64

## Add locations identifiers

In [8]:
# load location dictionary for matching names and codes
locations = pd.read_csv('./matching_data/locations.csv', encoding='utf-8', dtype=str)
locations

,regname,areaname,tername,KOATUU,KATOTTG,category,region_name
0,Запорізька область,Мелітопольський район,с.Терпіння,2323085101,UA23080270010078454,village,Zaporizka
1,Хмельницька область,Красилівський район,м.Красилів,6822710100,UA68040210010032567,town,Khmelnytska
2,Дніпропетровська область,Петропавлівський район,с.Дмитрівка,1223881501,UA12140170040016918,village,Dnipropetrovska
3,Чернівецька область,м.Чернівці,Шевченківський район міста,7310100000,UA73060610010033137,town,Chernivetska
4,Миколаївська область,Врадіївський район,с.Кумарі,4822383001,UA48080050190079797,village,Mykolaivska
...,...,...,...,...,...,...,...
30028,Одеська область,м.Південне,м.Південне,5111700000,UA51100410010044384,town,Odeska
30029,Нідерланди,м.Гаага,м.Гаага,0018030000,OC18030000000000000,abroad,Netherlands
30030,США,м.Воррен,м.Воррен,0026010000,OC26010000000000000,abroad,United States of America
30031,Польща,м.Домброва-Гурнича,м.Домброва-Гурнича,0021050000,OC21050000000000000,abroad,Poland


In [9]:
locations.rename(columns={'regname':'eoregname', 'areaname':'eoareaname', 'tername':'eotername'}, inplace=True)
locations.columns

Index(['eoregname', 'eoareaname', 'eotername', 'KOATUU', 'KATOTTG', 'category',
       'region_name'],
      dtype='object')

In [10]:
# add location codifier KOATUU
schools_edrpou_loc = schools_edrpou.merge(locations[['eoregname', 'eoareaname', 'eotername', 'KATOTTG', 'KOATUU', 'category']], on=['eoregname', 'eoareaname', 'eotername'], how='left')
schools_edrpou_loc

,eoname,eotypename,eoregname,eoareaname,eotername,eoedrpou,year,KATOTTG,KOATUU,category
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,Запорізька область,Мелітопольський район,с.Терпіння,NaN,2016,UA23080270010078454,2323085101,village
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,Хмельницька область,Красилівський район,м.Красилів,NaN,2016,UA68040210010032567,6822710100,town
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,Чернівецька область,м.Чернівці,Шевченківський район міста,NaN,2016,UA73060610010033137,7310100000,town
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,Донецька область,Донецька область,м.Дружківка,NaN,2016,UA14120030010055241,1411700000,town
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,Тернопільська область,Тернопільська область,м.Тернопіль,NaN,2016,UA61040490010069060,6110100000,town
...,...,...,...,...,...,...,...,...,...,...
99911,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,Львівська область,Стрийський район,м.Ходорів,05393837,2025,UA46100270010046817,4621510500,town
99912,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,м.Київ,м.Київ,Дарницький район міста,42372041,2025,UA80000000000210193,8036300000,city
99913,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,Одеська область,Роздільнянський район,с.Гаївка,25038104,2025,UA51140150050060575,5123981101,village
99914,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,Житомирська область,Житомирський район,с.Вільха,45498834,2025,UA18040470090069061,1821481001,village


In [11]:
# check whether there are NaN in KOATUU
schools_edrpou_loc[schools_edrpou_loc.KATOTTG.isna()]

,eoname,eotypename,eoregname,eoareaname,eotername,eoedrpou,year,KATOTTG,KOATUU,category
8484,"Лучицький навчально-виховний комплекс ""Загальн...",навчально-виховний комплекс,Львівська область,Львівська область,Сокальський район,NaN,2016,NaN,NaN,NaN


In [12]:
schools_edrpou_loc[schools_edrpou_loc.KATOTTG.isna()].eoname.unique()[0]

'Лучицький навчально-виховний комплекс "Загальноосвітня школа І-ІІІ ступенів - дитячий садок" Сокальської районної ради Львівської області\r\n'

In [13]:
to_set = schools_edrpou_loc[schools_edrpou_loc.KATOTTG.isna()].eoname.unique()[0]
schools_edrpou_loc.loc[schools_edrpou_loc.eoname == to_set, 'KATOTTG'] = 'UA46120110270041867'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname == to_set, 'KOATUU'] = '4624883201'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname == to_set, 'eoedrpou'] = '23947156'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname == to_set, 'category'] = 'village'
schools_edrpou_loc[schools_edrpou_loc.eoname == to_set]


,eoname,eotypename,eoregname,eoareaname,eotername,eoedrpou,year,KATOTTG,KOATUU,category
8484,"Лучицький навчально-виховний комплекс ""Загальн...",навчально-виховний комплекс,Львівська область,Львівська область,Сокальський район,23947156,2016,UA46120110270041867,4624883201,village


In [14]:
# drop names of region, area and territory
schools_edrpou_loc = schools_edrpou_loc[['eoname', 'eotypename', 'eoedrpou', 'year', 'KATOTTG', 'KOATUU', 'category']]
schools_edrpou_loc.drop_duplicates(inplace=True)
schools_edrpou_loc 

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/4162456799.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schools_edrpou_loc.drop_duplicates(inplace=True)


,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,NaN,2016,UA23080270010078454,2323085101,village
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,NaN,2016,UA68040210010032567,6822710100,town
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,NaN,2016,UA73060610010033137,7310100000,town
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,NaN,2016,UA14120030010055241,1411700000,town
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,NaN,2016,UA61040490010069060,6110100000,town
...,...,...,...,...,...,...,...
99911,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town
99912,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city
99913,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village
99914,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village


In [21]:
# check whether there are duplicates by name and location
schools_edrpou_loc[schools_edrpou_loc.duplicated(subset=['eoname', 'year', 'KATOTTG'], keep=False)]

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category


In [17]:
schools_edrpou_loc[schools_edrpou_loc.duplicated(subset=['eoname', 'year', 'KATOTTG'], keep=False)].eoname.unique()

array(['ДОНЕЦЬКЕ ВИЩЕ УЧИЛИЩЕ ОЛІМПІЙСЬКОГО РЕЗЕРВУ ІМ.С.БУБКИ',
       'Відокремлений структурний підрозділ "Олімпійський фаховий коледж імені Івана Піддубного Національного університету фізичного виховання і спорту України"',
       'Данилівський заклад загальної середньої освіти І-ІІІ ступенів Хустської міської ради Закарпатської області',
       'Комунальний заклад фахової передвищої освіти "Миколаївський фаховий коледж фізичної культури" Миколаївської обласної ради',
       'Ставищенський ліцей №1 Ставищенської селищної ради Білоцерківського району Київської області',
       'Лохвицька гімназія №1 Лохвицької міської ради Полтавської області',
       'Комунальний заклад вищої освіти Львівської обласної ради "Львівська медична академія імені Андрея Крупинського"'],
      dtype=object)

In [18]:
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='ДОНЕЦЬКЕ ВИЩЕ УЧИЛИЩЕ ОЛІМПІЙСЬКОГО РЕЗЕРВУ ІМ.С.БУБКИ', 'eotypename'] = 'заклад фахової передвищої освіти'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Відокремлений структурний підрозділ "Олімпійський фаховий коледж імені Івана Піддубного Національного університету фізичного виховання і спорту України"', 'eotypename'] = 'заклад фахової передвищої освіти'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Данилівський заклад загальної середньої освіти І-ІІІ ступенів Хустської міської ради Закарпатської області', 'eotypename'] = 'навчально-виховний комплекс'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Комунальний заклад фахової передвищої освіти "Миколаївський фаховий коледж фізичної культури" Миколаївської обласної ради', 'eotypename'] = 'заклад фахової передвищої освіти'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Ставищенський ліцей №1 Ставищенської селищної ради Білоцерківського району Київської області', 'eotypename'] = 'ліцей'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Лохвицька гімназія №1 Лохвицької міської ради Полтавської області', 'eotypename'] = 'гімназія'
schools_edrpou_loc.loc[schools_edrpou_loc.eoname=='Комунальний заклад вищої освіти Львівської обласної ради "Львівська медична академія імені Андрея Крупинського"', 'eotypename'] = 'заклад вищої освіти'

In [19]:
schools_edrpou_loc.drop_duplicates(inplace=True)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/55812111.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schools_edrpou_loc.drop_duplicates(inplace=True)


In [20]:
schools_edrpou_loc.shape

(99907, 7)

In [22]:
# check whether there are duplicates by name and location
schools_edrpou_loc[schools_edrpou_loc.duplicated(subset=['eoname', 'year', 'KATOTTG'], keep=False)]

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category


## Create id

In [23]:
create_id(schools_edrpou_loc, 'eoname', 'id')

/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:165: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.loc[:,id_] = dataset[attr]
/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:180: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset[id_] = dataset[id_].apply(del_rayon)
/Users/scipyguru/Library/Mobile Documents/com~apple~Clou

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,NaN,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,NaN,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,NaN,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,NaN,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,NaN,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...
...,...,...,...,...,...,...,...,...
99911,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...
99912,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб
99913,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...
99914,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...


In [24]:
# Скільки значень заповнилось
total = schools_edrpou_loc.shape[0]
filled = schools_edrpou_loc['eoedrpou'].notna().sum()
print(f"Filled: {filled} out of {total}  or {round(filled/total*100)}%")

Filled: 8468 out of 99907  or 8%


In [25]:
schools_edrpou_loc[schools_edrpou_loc['eoedrpou'].isna()].year.value_counts()

year
2018    11001
2019    10875
2020    10743
2016    10491
2021    10467
2017    10081
2022     9752
2023     9202
2024     8826
2025        1
Name: count, dtype: int64

In [26]:
schools_edrpou_loc[(schools_edrpou_loc['eoedrpou'].isna())&(schools_edrpou_loc['year'] == 2025)]

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
94273,Миколаївський юридичний фаховий коледж Націона...,заклад вищої освіти,NaN,2025,UA48060150010139573,4810136300,city,миколавськийюридичнийколеджнацональногоунверси...


In [27]:
schools_edrpou_loc[(schools_edrpou_loc['eoedrpou'].isna())&(schools_edrpou_loc['year'] == 2025)].eoname.unique()

array(['Миколаївський юридичний фаховий коледж Національного університету "Одеська юридична академія"'],
      dtype=object)

In [28]:
schools_edrpou_loc.loc[(schools_edrpou_loc['eoedrpou'].isna())&(schools_edrpou_loc['year'] == 2025),'eoedrpou' ]='20933314_'

## Fill from 2025 edrpou

In [29]:
assert (
    schools_edrpou_loc
    .dropna(subset=['eoedrpou'])
    .groupby(['id', 'KATOTTG'])['eoedrpou']
    .nunique()
    .le(1)
    .all()
)

In [30]:
schools_edrpou_loc['eoedrpou'] = schools_edrpou_loc.groupby(['id', 'KATOTTG'])['eoedrpou'].transform(
    lambda x: x.ffill().bfill()
)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/1122107747.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()
/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/1122107747.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schools_edrpou_loc['eoedrpou'] = schools_edrpou_loc.groupby(['id', 'KATOTTG'])['eoedrpou'].transform(


In [31]:
# Скільки значень заповнилось
filled =schools_edrpou_loc['eoedrpou'].notna().sum()
print(f"Filled: {filled} out of {total}  or {round(filled/total*100)}%")

Filled: 34343 out of 99907  or 34%


In [32]:
schools_edrpou_loc[schools_edrpou_loc['eoedrpou'].isna()].year.value_counts()

year
2016    9952
2018    9925
2019    9468
2017    9458
2020    9040
2021    7966
2022    5608
2023    2778
2024    1369
Name: count, dtype: int64

# Schools 2026

In [33]:
schools_2026 =  pd.read_excel('./school_data/schools_data_gov_ua_2026_02.xlsx', dtype='str')
schools_2026

,university_name,edrpou,university_type_name,education_type_name,university_level,location_type,university_financing_type_name,post_index,koatuu_id,language,...,university_address,post_index_u,koatuu_id_u,region_name_u,koatuu_name_u,university_address_u,university_phone,university_email,university_site,university_director_fio
0,Бердянська загальноосвітня школа І-ІІІ ступені...,26338802,заклад середньої освіти,загальносвітня школа,I-III,МІСТО,комунальна,71117,2310400000,! not applicable,...,"вул. Херсонська, 1/90,",71117,2310400000,Запорізька область,Бердянськ,"вул. Херсонська, 1/90,","(06153)77-270, (06153)78-270",brd-school7@ukr.net,http://brd-school7.ucoz.ua,! not applicable
1,"Первомайський навчально виховний комплекс ""Заг...",25992635,заклад середньої освіти,навчально-виховний комплекс (об'єднання),I-III,МІСТО,комунальна,55202,4810400000,! not applicable,...,"вул. Коротченко, 18-а",55202,4810400000,Миколаївська область,Первомайськ,"вул. Коротченко, 18-а",0516174548,NVK-15@i.ua,NaN,! not applicable
2,Загальноосвітня школа І-ІІІ ступенів №9 Мирног...,25689474,заклад середньої освіти,загальносвітня школа,I-III,МІСТО,комунальна,85321,1411300000,! not applicable,...,"мрн. Західниий, 4а",85321,1411300000,Донецька область,Мирноград,"мрн. Західниий, 4а",(06239)64352,osh9mirnograd@gmail.com,http://osh9.at.ua,! not applicable
3,Заклад загальної середньої освіти І-ІІ ступені...,32624238,заклад середньої освіти,ліцей,I-III,МІСТО,комунальна,85323,1411300000,! not applicable,...,"вул. Карбишева, 2",85323,1411300000,Донецька область,Мирноград,"вул. Карбишева, 2",(06239)64541,dimlicei04@ukr.net,http://lyceum0918.wixsite.com/harmony/,! not applicable
4,"Комунальний заклад ""Середня загальноосвітня шк...",23370351,заклад середньої освіти,загальносвітня школа,I-III,МІСТО,комунальна,51900,1210436900,! not applicable,...,"вул. Братська, 4",51900,1210436900,Дніпропетровська область,"Кам'янське, Заводський район","вул. Братська, 4",(05692)33148,school23drdn@ukr.net,http://shkola232016.ucoz.site/,! not applicable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15256,Приватний заклад загальної середньої освіти ``...,42988398,заклад середньої освіти,загальносвітня школа,II-III,МІСТО,приватна,NaN,4610137500,! not applicable,...,"ал. Замарстинівська, 83а",NaN,4610137500,Львівська область,"Львів, Шевченківський район","ал. Замарстинівська, 83а",NaN,NaN,NaN,! not applicable
15257,"ТОВ приватний заклад освіти початкова школа ""Е...",42973250,заклад середньої освіти,загальносвітня школа,I,МІСТО,приватна,02152,8036600000,! not applicable,...,"вул. Регенаторна, 4",02152,8036600000,м.Київ,"Київ, Дніпровський район","вул. Регенаторна, 4",(097)4102208,bfine1867@ukr.net,NaN,! not applicable
15258,Львівська загальоосвітня школа ПП Стембрідж Скул,42998605,заклад середньої освіти,загальносвітня школа,I-III,МІСТО,приватна,NaN,4610137500,! not applicable,...,"вул. Клепарівська, 30",NaN,4610137500,Львівська область,"Львів, Шевченківський район","вул. Клепарівська, 30",(096)2751027,NaN,NaN,! not applicable
15259,"ПРИВАТНИЙ ЗАКЛАД ""ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬОЇ О...",43128932,заклад середньої освіти,загальносвітня школа,I,МІСТО,приватна,08200,3210900000,! not applicable,...,"вул. Полтавська, 29 Д",08200,3210900000,Київська область,Ірпінь,"вул. Полтавська, 29 Д",(097)0059090,abetka123@email.ua,NaN,! not applicable


In [34]:
schools_2026 = schools_2026[['university_name', 'edrpou', 'koatuu_id']]

In [35]:
schools_2026.rename(columns={'edrpou':'eoedrpou', 'koatuu_id':'KOATUU'}, inplace=True)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/558460108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  schools_2026.rename(columns={'edrpou':'eoedrpou', 'koatuu_id':'KOATUU'}, inplace=True)


In [36]:
create_id(schools_2026, 'university_name', 'id')

/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:165: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.loc[:,id_] = dataset[attr]
/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:180: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset[id_] = dataset[id_].apply(del_rayon)
/Users/scipyguru/Library/Mobile Documents/com~apple~Clou

,university_name,eoedrpou,KOATUU,id
0,Бердянська загальноосвітня школа І-ІІІ ступені...,26338802,2310400000,бердянськазагальноосвтняшкола3ступ7бердянськаз...
1,"Первомайський навчально виховний комплекс ""Заг...",25992635,4810400000,первомайськийнавчальновиховнийкомплексзагально...
2,Загальноосвітня школа І-ІІІ ступенів №9 Мирног...,25689474,1411300000,загальноосвтняшкола3ступ9мирноградськадонецькаобл
3,Заклад загальної середньої освіти І-ІІ ступені...,32624238,1411300000,2ступлцейгармонямирноградськадонецькаобл
4,"Комунальний заклад ""Середня загальноосвітня шк...",23370351,1210436900,середнязагальноосвтняшкола23камянськао
...,...,...,...,...
15256,Приватний заклад загальної середньої освіти ``...,42988398,4610137500,приватнийтстепску
15257,"ТОВ приватний заклад освіти початкова школа ""Е...",42973250,8036600000,приватнийзакладосвтипочатковашколаеврик
15258,Львівська загальоосвітня школа ПП Стембрідж Скул,42998605,4610137500,льввськазагальоосвтняшколаппстембрджску
15259,"ПРИВАТНИЙ ЗАКЛАД ""ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬОЇ О...",43128932,3210900000,приватнийзакладабеткао


In [37]:
schools_2026 = schools_2026[schools_2026.eoedrpou.notna()]

In [38]:
schools_2026[schools_2026.duplicated(subset=['id', 'KOATUU'], keep=False)]

,university_name,eoedrpou,KOATUU,id
648,Рафалівська ЗОШ ІІ-ІІІ ступенів Володимирецько...,22567925,5620855400,рафалвськазагальноосвтняшкола3ступволодимирець...
683,Рафалівська загальноосвітня школа І-ІІІ ступен...,04590760,5620855400,рафалвськазагальноосвтняшкола3ступволодимирець...
12015,Червенівська загальноосвітня школа І ступеня М...,3491927,2122782805,червенвськазагальноосвтняшколаступмукачвськаза...
12040,Червенівська загальноосвітня школа І-І ступені...,34855921,2122782805,червенвськазагальноосвтняшколаступмукачвськаза...


In [39]:

schools_2026 = schools_2026[~schools_2026.eoedrpou.isin(['22567925', '04590760', '3491927', 
                                                                '34855921'])]

In [40]:
schools_ed2026 = merging_edrpou(schools_edrpou_loc, schools_2026[['KOATUU', 'id', 'eoedrpou']], ['KOATUU', 'id'])
schools_ed2026 

Filled: 63498 out of 99907  or 64%


,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,26373098,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25880114,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,NaN,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,25705061,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,14040173,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...
...,...,...,...,...,...,...,...,...
99902,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...
99903,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб
99904,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...
99905,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...


In [41]:
schools_ed2026[schools_ed2026 ['eoedrpou'].isna()].year.value_counts()

year
2016    6469
2018    5299
2017    5291
2019    4342
2021    4213
2020    3882
2022    3754
2023    2008
2024    1151
Name: count, dtype: int64

# Youcontrol

In [42]:
youcontrol =  pd.read_excel('./school_data/Youcontrol_historical_name.xlsx', dtype='str')
youcontrol 

,Код ЄДРПОУ,Назва,Коротка назва
0,00121146,"ВІДКРИТЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...",NaN
1,00121146,"ПУБЛІЧНЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...","АТ ""ПТЕМ"""
2,00121146,"ПРИВАТНЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...","АТ ""ПТЕМ"""
3,00131446,"КОЛЕКТИВНЕ ПІДПРИЄМСТВО ""БІЛГОРОД-ДНІСТРОВСЬКА...",NaN
4,00131446,ІНШІ ОРГАНІЗАЦІЙНО-ПРАВОВІ ФОРМИ КОЛЕКТИВНЕ ПІ...,NaN
...,...,...,...
80921,60043961,ДЕРЕБЧИНСЬКИЙ ОПОРНИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬ...,ДОЗЗСО
80922,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ"""
80923,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ"""
80924,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ"""


In [43]:
youcontrol.columns

Index(['Код ЄДРПОУ', 'Назва', 'Коротка назва'], dtype='object')

## long name

In [44]:
youcontrol[youcontrol.duplicated(subset=['Назва', 'Код ЄДРПОУ'], keep=False)]

,Код ЄДРПОУ,Назва,Коротка назва
3,00131446,"КОЛЕКТИВНЕ ПІДПРИЄМСТВО ""БІЛГОРОД-ДНІСТРОВСЬКА...",NaN
5,00131446,"КОЛЕКТИВНЕ ПІДПРИЄМСТВО ""БІЛГОРОД-ДНІСТРОВСЬКА...",КП МК-26
119,00452713,"ДЕРЖАВНИЙ НАВЧАЛЬНИЙ ЗАКЛАД ""ЗВЕНИГОРОДСЬКИЙ Ц...","ДНЗ ""ЗВЕНИГОРОДСЬКИЙ ЦППРК"""
121,00452713,"ДЕРЖАВНИЙ НАВЧАЛЬНИЙ ЗАКЛАД ""ЗВЕНИГОРОДСЬКИЙ Ц...","ДНЗ ""ЗВЕНИГОРОДСЬКИЙ ЦППРК"""
144,00854423,"КООПЕРАТИВ ""НОВОКРОПИВНИЦЬКИЙ""",NaN
...,...,...,...
80918,60043961,ДЕРЕБЧИНСЬКИЙ ОПОРНИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬ...,ДОЗЗСО
80919,60043961,ДЕРЕБЧИНСЬКА СЕРЕДНЯ ЗАГАЛЬНООСВІТНЯ ШКОЛА I-I...,ДЕРЕБЧИНСЬКА СЗШ I-III СТУПЕНІВ
80921,60043961,ДЕРЕБЧИНСЬКИЙ ОПОРНИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬ...,ДОЗЗСО
80922,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ"""


In [45]:
youcontrol_long = youcontrol.drop_duplicates(subset=['Назва', 'Код ЄДРПОУ'], keep='first')

In [46]:
create_id(youcontrol_long, 'Назва', 'id')

/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:165: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset.loc[:,id_] = dataset[attr]
/Users/scipyguru/Library/Mobile Documents/com~apple~CloudDocs/Documents_New/ZNO-Dataset/notebooks/tables_creation/src/merging_with_id.py:180: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset[id_] = dataset[id_].apply(del_rayon)
/Users/scipyguru/Library/Mobile Documents/com~apple~Clou

,Код ЄДРПОУ,Назва,Коротка назва,id
0,00121146,"ВІДКРИТЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...",NaN,вдкритеакцонернетовариствопвдентеплоенергомонта
1,00121146,"ПУБЛІЧНЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...","АТ ""ПТЕМ""",публчнеакцонернетовариствопвдентеплоенергомонта
2,00121146,"ПРИВАТНЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ПІВДЕНТЕПЛОЕНЕ...","АТ ""ПТЕМ""",приватнеакцонернетовариствопвдентеплоенергомонта
3,00131446,"КОЛЕКТИВНЕ ПІДПРИЄМСТВО ""БІЛГОРОД-ДНІСТРОВСЬКА...",NaN,колективнепдпримствоблгородднстровськамеханзов...
4,00131446,ІНШІ ОРГАНІЗАЦІЙНО-ПРАВОВІ ФОРМИ КОЛЕКТИВНЕ ПІ...,NaN,колективнепдпримствоблгородднстровськамеханзов...
...,...,...,...,...
80918,60043961,ДЕРЕБЧИНСЬКИЙ ОПОРНИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬ...,ДОЗЗСО,деребчинськийопорний3ступвницькаобл
80920,60043961,ДЕРЕБЧИНСЬКИЙ ОПОРНИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬ...,ДОЗЗСО,деребчинськийопорний3ступвницькаобл
80922,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ""",опорнийзакладдеребчинський3ступджуринськавниць...
80923,60043961,"ОПОРНИЙ ЗАКЛАД ""ДЕРЕБЧИНСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ...","ОЗ ""ДЕРЕБЧИНСЬКИЙ ЗЗСО I-III СТУПЕНІВ""",опорнийзакладдеребчинський3ступджуринськавниць...


In [47]:
youcontrol_long.rename(columns={'Код ЄДРПОУ':'eoedrpou'}, inplace=True)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/3977263988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  youcontrol_long.rename(columns={'Код ЄДРПОУ':'eoedrpou'}, inplace=True)


In [48]:
youcontrol_long[(youcontrol_long.eoedrpou.str.len() < 8)&(youcontrol_long.eoedrpou.str.len().notna())]

,eoedrpou,Назва,Коротка назва,id


In [49]:
youcontrol_long = youcontrol_long.drop_duplicates(subset=['id', 'eoedrpou'], keep='first')

In [50]:
youcontrol_long[youcontrol_long.duplicated(subset=['id'], keep=False)]

,eoedrpou,Назва,Коротка назва,id
221,01284979,МІЖШКІЛЬНИЙ НАВЧАЛЬНО-ВИРОБНИЧИЙ КОМБІНАТ,МНВК,мжшкльнийнавчальновиробничийкомбна
494,02218789,"КОМУНАЛЬНА ОРГАНІЗАЦІЯ (УСТАНОВА, ЗАКЛАД) МУЗИ...",NaN,музичнашкола2
498,02218884,"КОМУНАЛЬНА ОРГАНІЗАЦІЯ (УСТАНОВА, ЗАКЛАД) КОМУ...",NaN,музичнашкола3
503,02218921,"КОМУНАЛЬНА ОРГАНІЗАЦІЯ (УСТАНОВА, ЗАКЛАД) КОМУ...",NaN,музичнашкола2
719,02542129,МИКОЛАЇВСЬКИЙ ПРОФЕСІЙНИЙ ЛІЦЕЙ,NaN,миколавськийпрофесйнийлце
...,...,...,...,...
80889,45117367,"ФЕРМЕРСЬКЕ ГОСПОДАРСТВО ""ДОВІРА Б""","ФГ ""ДОВІРА Б""",фермерськегосподарстводовра
80903,45251511,Юріївська гімназія з початковою школою та дошк...,Юріївська гімназія,юрвськагмназязпочатковоюшколоютадошкльнимпдроз...
80905,45275530,Юріївська гімназія з початковою школою та дошк...,Юріївська гімназія,юрвськагмназязпочатковоюшколоютадошкльнимпдроз...
80907,45287518,Кобзарцівська початкова школа з дошкільним під...,Кобзарцівська початкова школа,кобзарцвськапочатковашколаздошкльнимпдроздломс...


In [51]:
youcontrol_long.drop(youcontrol_long[youcontrol_long.duplicated(subset=['id'], keep=False)].index.values.tolist()  , inplace=True)

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/2842222147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  youcontrol_long.drop(youcontrol_long[youcontrol_long.duplicated(subset=['id'], keep=False)].index.values.tolist()  , inplace=True)


In [52]:
schools_youcontrol_long = merging_edrpou(schools_ed2026, youcontrol_long[['id', 'eoedrpou']], ['id'])
schools_youcontrol_long

Filled: 85675 out of 99907  or 86%


,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,26373098,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25880114,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,21431046,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,25705061,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,14040173,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...
...,...,...,...,...,...,...,...,...
99902,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...
99903,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб
99904,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...
99905,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...


In [53]:
schools_youcontrol_long[schools_youcontrol_long['eoedrpou'].isna()].year.value_counts()

year
2016    2819
2018    2337
2017    2077
2019    1946
2020    1698
2021    1403
2022    1123
2023     513
2024     316
Name: count, dtype: int64

# Gathered by hand

In [54]:
data_hand = pd.read_csv('./school_data/by_hand.csv',  dtype = str)
data_hand

,eoname,KOATUU_2020,KOATUU_2020_reg,year,id,EDRPOU
0,"Вiдокремлений структурний пiдроздiл ""Краматорс...",0510100000,05,[2023],краматорськийколеджпромисловостнформацйнихтехн...,04601943
1,"ВСП ""Технологічно-промисловий фаховий коледж В...",0510100000,05,[2022],технологчнопромисловийколеджвницькогонацональн...,00419667
2,Вище професійне училище №11 м. Вінниці,0510100000,05,"[2018, 2019, 2020, 2021, 2022, 2023]",вищепрофесйнеучилище11мвниц,03065891
3,Вище художнє професійно-технічне училище № 5 м...,0510100000,05,[2023],вищехудожнпрофесйнотехнчнеучилище5мвниц,02539890
4,Вище художнє професійно-технічне училище №5 м....,0510100000,05,"[2018, 2019, 2020, 2021, 2022]",вищехудожнпрофесйнотехнчнеучилище5мвниця,02539890
...,...,...,...,...,...,...
32052,Школа №25 І-ІІІ ступенів Шевченківського район...,8039100000,80,"[2016, 2017, 2018, 2019, 2020, 2021]",школа253ступмкива,22880786
32053,Школа №27 І-ІІІ ступенів Шевченківського район...,8039100000,80,[2016],школа273ступмкива,26125710
32054,Школа №70 І-ІІІ ступенів Шевченківського район...,8039100000,80,[2016],школа703ступмкива,22881828
32055,Школа №95 І-ІІІ ступенів Шевченківського район...,8039100000,80,[2017],школа953ступмкива,26125905


In [55]:
data_hand.columns

Index(['eoname', 'KOATUU_2020', 'KOATUU_2020_reg', 'year', 'id', 'EDRPOU'], dtype='object')

In [56]:
data_hand = data_hand[['eoname', 'KOATUU_2020', 'EDRPOU']]

In [57]:
data_hand['temp_eoname'] = data_hand['eoname'].str.replace(r'\r\n|\n', '', regex=True).str.strip()

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/1678710856.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_hand['temp_eoname'] = data_hand['eoname'].str.replace(r'\r\n|\n', '', regex=True).str.strip()


In [58]:
data_hand.rename(columns={'EDRPOU':'eoedrpou', 'KOATUU_2020': 'KOATUU'}, inplace=True)
data_hand.columns

/var/folders/cz/wq9d8j_11fx3b8pjjbk7z8br0000gn/T/ipykernel_65500/1071490741.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_hand.rename(columns={'EDRPOU':'eoedrpou', 'KOATUU_2020': 'KOATUU'}, inplace=True)


Index(['eoname', 'KOATUU', 'eoedrpou', 'temp_eoname'], dtype='object')

In [59]:
data_hand[data_hand.duplicated(subset=['temp_eoname', 'eoedrpou', 'KOATUU'], keep=False)]

,eoname,KOATUU,eoedrpou,temp_eoname
483,Степенівський навчально - виховний комплекс: з...,0520687603,23562485,Степенівський навчально - виховний комплекс: з...
484,Степенівський навчально - виховний комплекс: з...,0520687603,23562485,Степенівський навчально - виховний комплекс: з...
619,"Комунальний заклад ""Сербинівський навчально-ви...",0521085403,39977763,"Комунальний заклад ""Сербинівський навчально-ви..."
620,"Комунальний заклад ""Сербинівський навчально-ви...",0521085403,39977763,"Комунальний заклад ""Сербинівський навчально-ви..."
1145,"Комунальний заклад ""Воловодівська загальноосві...",0523084004,25918745,"Комунальний заклад ""Воловодівська загальноосві..."
...,...,...,...,...
26594,Новосеменівська загальноосвітня школа I-III ст...,6522955112,24750102,Новосеменівська загальноосвітня школа I-III ст...
29182,Лип'янська загальноосвітня школа І-ІІІ ступені...,7125784001,24355707,Лип'янська загальноосвітня школа І-ІІІ ступені...
29184,Лип'янська загальноосвітня школа І-ІІІ ступені...,7125784001,24355707,Лип'янська загальноосвітня школа І-ІІІ ступені...
31260,Школа І-ІІІ ступенів №284 Дарницького району м...,8036300000,22875957,Школа І-ІІІ ступенів №284 Дарницького району м...


In [60]:
data_hand = data_hand.drop_duplicates(subset=['temp_eoname', 'eoedrpou', 'KOATUU'], keep='first')

In [61]:
data_hand[data_hand.duplicated(subset=['temp_eoname', 'KOATUU'], keep=False)]

,eoname,KOATUU,eoedrpou,temp_eoname
19465,Розівська загальноосвітня школа I-III ступенів...,5124585101,26506286,Розівська загальноосвітня школа I-III ступенів...
19466,Розівська загальноосвітня школа I-III ступенів...,5124585101,25814353,Розівська загальноосвітня школа I-III ступенів...


In [62]:
data_hand = data_hand.drop_duplicates(subset=['temp_eoname', 'KOATUU'], keep='first')

In [63]:
schools_youcontrol_long['temp_eoname'] = schools_youcontrol_long['eoname'].str.replace(r'\r\n|\n', '', regex=True).str.strip()
schools_youcontrol_long

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id,temp_eoname
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,26373098,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...,"Терпіннівський колегіум ""Джерело"" Мелітопольсь..."
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25880114,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...,Красилівська загальноосвітня школа I-III ступе...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,21431046,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...,Чернівецька спеціалізована школа І-ІІІ ступені...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,25705061,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл,Загальноосвітня школа I-III ступенів № 6 Дружк...
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,14040173,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...,Тернопільська спеціалізована школа І-ІІІ ступе...
...,...,...,...,...,...,...,...,...,...
99902,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...,"Ходорівське відділення ВСП ""Технічний фаховий ..."
99903,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА..."
99904,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...,"Гаївська філія ""Початкова школа-заклад дошкіль..."
99905,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...,Вільшанська філія Романівського ліцею №2 Роман...


In [64]:
schools_final = merging_edrpou(schools_youcontrol_long, data_hand[['temp_eoname', 'eoedrpou', 'KOATUU']], ['temp_eoname', 'KOATUU'])
schools_final

Filled: 99651 out of 99907  or 100%


,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id,temp_eoname
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,26373098,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...,"Терпіннівський колегіум ""Джерело"" Мелітопольсь..."
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25880114,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...,Красилівська загальноосвітня школа I-III ступе...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,21431046,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...,Чернівецька спеціалізована школа І-ІІІ ступені...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,25705061,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл,Загальноосвітня школа I-III ступенів № 6 Дружк...
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,14040173,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...,Тернопільська спеціалізована школа І-ІІІ ступе...
...,...,...,...,...,...,...,...,...,...
99902,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...,"Ходорівське відділення ВСП ""Технічний фаховий ..."
99903,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА..."
99904,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...,"Гаївська філія ""Початкова школа-заклад дошкіль..."
99905,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...,Вільшанська філія Романівського ліцею №2 Роман...


In [65]:
schools_final.drop(columns=['temp_eoname'], inplace=True)

In [66]:
schools_final

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
0,"Терпіннівський колегіум ""Джерело"" Мелітопольсь...",колегіум,26373098,2016,UA23080270010078454,2323085101,village,терпнвськийколегумджереломелтопольськазапорзьк...
1,Красилівська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25880114,2016,UA68040210010032567,6822710100,town,красилвськазагальноосвтняшкола3ступ3хмельницьк...
2,Чернівецька спеціалізована школа І-ІІІ ступені...,спеціалізована школа,21431046,2016,UA73060610010033137,7310100000,town,чернвецькаспецалзованашкола3ступфзикоматематич...
3,Загальноосвітня школа I-III ступенів № 6 Дружк...,середня загальноосвітня школа,25705061,2016,UA14120030010055241,1411700000,town,загальноосвтняшкола3ступ6дружквськадонецькаобл
4,Тернопільська спеціалізована школа І-ІІІ ступе...,спеціалізована школа,14040173,2016,UA61040490010069060,6110100000,town,тернопльськаспецалзованашкола3ступ3зпоглиблени...
...,...,...,...,...,...,...,...,...
99902,"Ходорівське відділення ВСП ""Технічний фаховий ...",заклад фахової передвищої освіти,05393837,2025,UA46100270010046817,4621510500,town,ходорвськевддленятехнчнийколеджнацональногоунв...
99903,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРИВА...",гімназія,42372041,2025,UA80000000000210193,8036300000,city,приватнийзакладосвтигмназякларсверб
99904,"Гаївська філія ""Початкова школа-заклад дошкіль...",початкова школа,25038104,2025,UA51140150050060575,5123981101,village,гавськафляпочатковашколазакладдошкльнаосвтияко...
99905,Вільшанська філія Романівського ліцею №2 Роман...,гімназія,45498834,2025,UA18040470090069061,1821481001,village,вльшанськафляроманвськоголцею2романвськажитоми...


In [67]:
schools_final[schools_final.eoedrpou.isna()].year.value_counts()

year
2024    200
2016     13
2017     11
2019      9
2018      8
2020      6
2021      6
2022      2
2023      1
Name: count, dtype: int64

In [69]:
schools_final.to_csv('./final_tables/schools_edrpou_to_fix.csv', index = False)

In [68]:
schools_final[(schools_final.eoedrpou.isna())&(schools_final.year==2023)]

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
75288,"Відокремлений структурний підрозділ ""Донбаськи...",заклад фахової передвищої освіти,NaN,2023,UA14120210010032554,1414100000,town,донбаськийаграрнийколеджсхдноукранськогонацона...


In [ ]:
schools_final[(schools_final.eoedrpou.isna())&(schools_final.year==2023)].eoname.unique()

In [70]:
mask = schools_final.groupby(['id', 'KATOTTG'])['eoedrpou'].transform('nunique') > 1

rows_with_diff_edrpou = schools_final[mask].sort_values(['id', 'KATOTTG', 'eoedrpou'])
rows_with_diff_edrpou

,eoname,eotypename,eoedrpou,year,KATOTTG,KOATUU,category,id
1720,Добросинська загальноосвітня школа І-ІІІ ступе...,середня загальноосвітня школа,22356737,2016,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
10980,Добросинська загальноосвітня школа І-ІІІ ступе...,середня загальноосвітня школа,22356737,2017,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
22899,Добросинська загальноосвітня школа І-ІІІ ступе...,середня загальноосвітня школа,22356737,2018,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
37087,Добросинська загальноосвітня школа І-ІІІ ступе...,середня загальноосвітня школа,22356737,2019,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
46801,Добросинська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25236844,2020,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
55077,Добросинська загальноосвітня школа I-III ступе...,середня загальноосвітня школа,25236844,2021,UA46060110010060620,4622783201,village,добросинськазагальноосвтняшкола3ступжовквськао
7554,"Пиріжнянський навчально-виховний комплекс ""Заг...",навчально-виховний комплекс,33775772,2016,UA51120090130079081,5122584101,village,пиржнянськийнавчальновиховнийкомплексзагальноо...
20021,"комунальний заклад ""Пиріжнянський навчально-ви...",навчально-виховний комплекс,33775788,2017,UA51120090130079081,5122584101,village,пиржнянськийнавчальновиховнийкомплексзагальноо...
13430,Спеціалізована школа-інтернат І-ІІІ ступенів ...,спеціалізована школа-інтернат,22357335,2017,UA46060390010035472,4622710400,town,спецалзованашколантернат3ступмраваруськ
22795,Спеціалізована школа-інтернат І-ІІІ ступенів м...,спеціалізована школа-інтернат,39229119,2018,UA46060390010035472,4622710400,town,спецалзованашколантернат3ступмраваруськ


In [ ]:
schools_final.loc[(schools_final.KATOTTG=='UA46060110010060620')&(schools_final.id=='добросинськазагальноосвтняшкола3ступжовквськао'), 'eoedrpou']='22356737'
schools_final.loc[(schools_final.KATOTTG=='UA51120090130079081')&(schools_final.id=='пиржнянськийнавчальновиховнийкомплексзагальноосвтняшкола3ступдошкльнийнавчальнийзакладкодимськаодеськаобл'), 'eoedrpou']='33775788'
schools_final.loc[(schools_final.KATOTTG=='UA46060390010035472')&(schools_final.id=='спецалзованашколантернат3ступмраваруськ'), 'eoedrpou']='22357335'
schools_final.loc[(schools_final.KATOTTG=='UA59100030010033445')&(schools_final.id=='сумськаобласнаглухвськазагальноосвтняшколантернат3ступменмжужом'), 'eoedrpou']='21128773'
schools_final.loc[(schools_final.KATOTTG=='UA35020190090043341')&(schools_final.id=='сухоташлицькийнавчальновиховнийкомплексзагальноосвтнйнавчальнийзаклад3ступдошкльнийнавчальнийзакладвльшанськао'), 'eoedrpou']='38337671'


In [ ]:
schools_final[schools_final.eoedrpou == '23056598']

In [ ]:
schools_final[schools_final.eoedrpou == '00727713']

# solve problems

In [ ]:
issues = pd.read_csv('./school_data/edrpou_problems.csv',  dtype = str)
issues = issues[issues.to_change.notna()]
issues = issues[['eoname', 'KOATUU_2020', 'EDRPOU', 'to_change']]
issues.drop_duplicates(inplace=True)
issues

In [ ]:
issues.rename(columns={'EDRPOU': 'eoedrpou', 'KOATUU_2020': 'KOATUU'}, inplace=True)
issues

In [ ]:
locations_unique= locations[['KATOTTG', 'KOATUU']].drop_duplicates(subset='KATOTTG')

In [ ]:
issues = issues.merge(locations_unique, on='KOATUU', how='left')
issues

In [ ]:
merged_df = schools_final.merge(issues[['eoname', 'eoedrpou', 'KATOTTG', 'to_change']], on=['eoname', 'eoedrpou', 'KATOTTG'], how='left')
print(merged_df.shape)
merged_df[merged_df['to_change'].notna()]

In [ ]:
merged_df.loc[merged_df['to_change'].notna(), 'eoedrpou'] = merged_df.loc[merged_df['to_change'].notna(), 'to_change']
merged_df[merged_df['to_change'].notna()]

In [ ]:
merged_df.shape

In [ ]:
merged_df = merged_df[['eoname', 'eotypename', 'KATOTTG', 'year', 'eoedrpou', 'id']]

In [ ]:
merged_df

In [ ]:
mask = merged_df.groupby(['id', 'KATOTTG'])['eoedrpou'].transform('nunique') > 1

rows_with_diff_edrpou = merged_df[mask].sort_values(['id', 'KATOTTG', 'eoedrpou'])
rows_with_diff_edrpou

In [ ]:
merged_df[merged_df.id=='виноградненськазагальноосвтняшкола3ступен'].eoedrpou.unique()

In [ ]:
schools_final[schools_final.id=='виноградненськазагальноосвтняшкола3ступен'].eoedrpou.unique()

In [ ]:
# find duplicates by type
duplicated = merged_df[[ 'eotypename', 'KOATUU_2020', 'year', 'eoedrpou']]
duplicated.drop_duplicates(inplace=True)
duplicated[duplicated.duplicated(subset=['KOATUU_2020', 'year', 'eoedrpou'], keep=False)].eoedrpou.unique()

In [ ]:
merged_df[merged_df.eoedrpou=='21065914'].eoname.unique()

In [ ]:
merged_df.loc[(merged_df.eoedrpou=='04544524')&(merged_df.year == 2023), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='20361881')&(merged_df.year.isin([2018, 2020, 2021, 2022])), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='25704593')&(merged_df.year == 2016)&(merged_df.eotypename=='вечірня (змінна) школа'), 'eoedrpou'] = '25704593a'
merged_df.loc[(merged_df.eoedrpou=='22067772')&(merged_df.year.isin([2023, 2021])), 'eotypename'] = 'ліцей'
merged_df.loc[(merged_df.eoname=='Філія "Світлодолинська гімназія" Терпіннівського опорного закладу загальної середньої освіти I-III ступенів "Джерело" Терпіннівської сільської ради Мелітопольського району Запорізької області')&(merged_df.year == 2022)&(merged_df.eotypename=='гімназія'), 'eoedrpou'] = '26373098a'
merged_df.loc[(merged_df.eoname=='Філія "Терпіннівська гімназія" Терпіннівського опорного закладу загальної середньої освіти I-III ступенів "Джерело"Терпіннівської сільської ради Мелітопольського району Запорізької області')&(merged_df.year == 2022)&(merged_df.eotypename=='гімназія'), 'eoedrpou'] = '26373098b'

merged_df.loc[(merged_df.eoedrpou=='26373098')&(merged_df.year == 2021), 'eotypename'] = 'колегіум'
merged_df.loc[merged_df.eoedrpou=='26317622', 'eotypename'] = 'спеціалізована школа'
merged_df.loc[(merged_df.eoedrpou=='26292402')&(merged_df.year == 2023)&(merged_df.eotypename=='початкова школа'), 'eoedrpou'] = '26292402a'
merged_df.loc[(merged_df.eoedrpou=='26292402')&(merged_df.year == 2017)&(merged_df.KOATUU_2020=='2325555137'), 'eoedrpou'] = '26292402a'
merged_df.loc[(merged_df.eoedrpou=='22208066')&(merged_df.year == 2023), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='25294847')&(merged_df.year == 2016), 'eotypename'] = 'середня загальноосвітня школа'
merged_df.loc[merged_df.eoedrpou=='34548369', 'eotypename'] = 'вечірня (змінна) школа'
merged_df.loc[(merged_df.eoedrpou=='33329158')&(merged_df.year == 2023), 'eotypename'] = 'ліцей'
merged_df.loc[(merged_df.eoedrpou=='43185063')&(merged_df.year == 2023)&(merged_df.eotypename=='гімназія'), 'eoedrpou'] = '43185063a'
merged_df.loc[(merged_df.eoedrpou=='23234551')&(merged_df.year == 2023)&(merged_df.eotypename=='початкова школа'), 'eoedrpou'] = '23234551a'
merged_df.loc[(merged_df.eoedrpou=='02136962')&(merged_df.year == 2023), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='36642931')&(merged_df.year.isin([2018, 2019, 2020])), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='36642931')&(merged_df.year == 2023)&(merged_df.eotypename=='середня загальноосвітня школа'), 'eoedrpou'] = '36642931a'
merged_df.loc[(merged_df.eoedrpou=='35657567')&(merged_df.year.isin([2018, 2019, 2020])), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='35657567')&(merged_df.year == 2023)&(merged_df.eotypename=='середня загальноосвітня школа'), 'eoedrpou'] = '35657567a'
merged_df.loc[(merged_df.eoname == 'Авіакосмічний ліцей №2 Національного авіаційного університету (м. Київ)'), 'eoedrpou'] = '26305984'
merged_df.loc[(merged_df.eoedrpou=='21065914')&(merged_df.year == 2023)&(merged_df.eotypename=='середня загальноосвітня школа'), 'eoedrpou'] = '21065914a'
merged_df.loc[(merged_df.eoedrpou=='24011586')&(merged_df.year == 2016)&(merged_df.eotypename=='вечірня (змінна) школа'), 'eoedrpou'] = '24011586a'
merged_df.loc[(merged_df.eoname == 'Відокремлений структурний підрозділ «Шосткинський професійний ліцей Сумського державного університету»'), 'eoedrpou'] = '39604993'
merged_df.loc[(merged_df.eoname == 'Писарівська загальноосвітня школа I-III ступенів')&(merged_df.KOATUU_2020=='0520685203'), 'eoedrpou'] = '26235048'
merged_df.loc[(merged_df.eoedrpou=='32535101')&(merged_df.year == 2023)&(merged_df.eotypename=='середня загальноосвітня школа'), 'eoedrpou'] = '32535101a'
merged_df.loc[(merged_df.eoedrpou=='14049079')&(merged_df.year == 2023)&(merged_df.eotypename=='гімназія'), 'eoedrpou'] = '14049079a'
merged_df.loc[(merged_df.eoedrpou=='23594835')&(merged_df.year == 2016), 'eotypename'] = "середня загальноосвітня школа"
merged_df.loc[(merged_df.eoedrpou=='04591392')&(merged_df.year == 2023)&(merged_df.eotypename=='спортивний ліцей'), 'eoedrpou'] = '04591392a'
merged_df.loc[(merged_df.eoedrpou=='05401005')&(merged_df.year == 2023), 'eotypename'] = 'заклад фахової передвищої освіти'
merged_df.loc[(merged_df.eoedrpou=='25780246')&(merged_df.year == 2023), 'eotypename'] = "ліцей"
merged_df.loc[(merged_df.eoedrpou=='24676096')&(merged_df.year == 2016), 'eotypename'] = "навчально-виховний комплекс"
merged_df.loc[merged_df.eoedrpou=='36045905', 'eotypename'] = 'вечірня (змінна) школа'
merged_df.loc[(merged_df.eoedrpou=='02011634')&(merged_df.eotypename=='ліцей'), 'eoedrpou'] = '02011634a'
merged_df.loc[merged_df.eoedrpou=='02011634', 'eotypename'] = 'заклад фахової передвищої освіти/ліцей'
merged_df.loc[(merged_df.eoname == 'Богданівська загальноосвітня школа I-III ступенів')&(merged_df.KOATUU_2020=='0524382201'), 'eoedrpou'] = '21723688'
merged_df.loc[(merged_df.eoedrpou=='13353993')&(merged_df.KOATUU_2020=='0720855700'), 'eoedrpou'] = '13353993a'
merged_df.loc[(merged_df.eoedrpou=='23019551')&(merged_df.KOATUU_2020=='0720582006'), 'eoedrpou'] = '23018936'
merged_df.loc[(merged_df.eoname == 'Криворізький юридичний коледж Національного університету "Одеська юридична академія"')&(merged_df.eoedrpou=='20933314'), 'eoedrpou'] = '20933314a'
merged_df.loc[(merged_df.eoname == 'Криворізький юридичний фаховий коледж Національного університету "Одеська юридична академія"')&(merged_df.eoedrpou=='20933314'), 'eoedrpou'] = '20933314a'
merged_df.loc[(merged_df.eoname == 'Миколаївський юридичний коледж Національного університету "Одеська юридична академія"')&(merged_df.eoedrpou=='20933314'), 'eoedrpou'] = '20933314b'
merged_df.loc[(merged_df.eoname == 'Миколаївський юридичний фаховий коледж Національного університету "Одеська юридична академія"')&(merged_df.eoedrpou=='20933314'), 'eoedrpou'] = '20933314b'
merged_df.loc[(merged_df.eoedrpou=='35061512')&(merged_df.KOATUU_2020=='1220755700'), 'eoedrpou'] = '23067521'
merged_df.loc[(merged_df.eoedrpou=='22054686')&(merged_df.KOATUU_2020=='1821484201'), 'eoedrpou'] = '22056461'
merged_df.loc[(merged_df.eoedrpou=='02543472')&(merged_df.KOATUU_2020=='1825455100'), 'eoedrpou'] = '02543472a'
merged_df.loc[(merged_df.eoedrpou=='02543472')&(merged_df.KOATUU_2020=='1825010100'), 'eoedrpou'] = '02543472b'
merged_df.loc[(merged_df.eoedrpou=='22058767')&(merged_df.KOATUU_2020=='2621685701'), 'eoedrpou'] = '23924178'
merged_df.loc[(merged_df.eoname == 'ДНЗ "Запорізьке вище професійне училище моди і стилю" (філія)')&(merged_df.eoedrpou=='02543880'), 'eoedrpou'] = '02543880a'
merged_df.loc[(merged_df.eoname == 'ФЕДОРІВСЬКИЙ ЦЕНТР ПРОФЕСІЙНОЇ ОСВІТИ (відділення в смт Більмак)')&(merged_df.eoedrpou=='02543822'), 'eoedrpou'] = '02543822a'
merged_df.loc[(merged_df.eoname == 'ДНЗ "Чубарівський центр професійно-технічної освіти" (відділення в смт Більмак)')&(merged_df.eoedrpou=='02543822'), 'eoedrpou'] = '02543822a'
merged_df.loc[(merged_df.eoedrpou=='02543822')&(merged_df.KOATUU_2020=='2321510100'), 'eoedrpou'] = '02543822b'
merged_df.loc[(merged_df.eoedrpou=='24618911')&(merged_df.KOATUU_2020=='2621684301'), 'eoedrpou'] = '23924215'
merged_df.loc[(merged_df.eoedrpou=='42408685')&(merged_df.KOATUU_2020=='3521181301'), 'eoedrpou'] = '42408685a'
merged_df.loc[(merged_df.eoedrpou=='42409406')&(merged_df.KOATUU_2020=='3521184401'), 'eoedrpou'] = '42409406a'
merged_df.loc[(merged_df.eoedrpou=='23228183')&(merged_df.KOATUU_2020=='3521780301'), 'eoedrpou'] = '23228183a'
merged_df.loc[(merged_df.eoedrpou=='33329158')&(merged_df.KOATUU_2020=='3523482402'), 'eoedrpou'] = '33329158a'
merged_df.loc[(merged_df.eoedrpou=='40779964')&(merged_df.KOATUU_2020=='3523686001'), 'eoedrpou'] = '40779964a'
merged_df.loc[(merged_df.eoedrpou=='40782500')&(merged_df.KOATUU_2020=='3523685601'), 'eoedrpou'] = '40782500a'
merged_df.loc[(merged_df.eoedrpou=='40782500')&(merged_df.KOATUU_2020=='35236818011'), 'eoedrpou'] = '23231848'
merged_df.loc[(merged_df.eoedrpou=='33249675')&(merged_df.KOATUU_2020=='3524986601'), 'eoedrpou'] = '33249675a'
merged_df.loc[(merged_df.eoname == 'Відокремлений підрозділ "Регіональний центр професійної освіти Луганського національного університету імені Тараса Шевченка" (Рубіжанське відділення)')&(merged_df.eoedrpou=='40180125'), 'eoedrpou'] = '40180125a'
merged_df.loc[(merged_df.eoname == 'Відокремлений підрозділ "Регіональний центр професійної освіти Луганського національного університету імені Тараса Шевченка" (Кремінське відділення)')&(merged_df.eoedrpou=='40180125'), 'eoedrpou'] = '40180125b'
merged_df.loc[(merged_df.eoname == 'Відокремлений підрозділ "Регіональний центр професійної освіти Луганського національного університету імені Тараса Шевченка" (Щастинське відділення)')&(merged_df.eoedrpou=='40180125'), 'eoedrpou'] = '40180125c'
merged_df.loc[(merged_df.eoname == 'Відокремлений підрозділ «Регіональний центр професійної освіти Луганського національного університету імені Тараса Шевченка» (Старобільське відділення)')&(merged_df.eoedrpou=='40180125'), 'eoedrpou'] = '40180125d'
merged_df.loc[(merged_df.eoedrpou=='05393837')&(merged_df.KOATUU_2020=='4621510500'), 'eoedrpou'] = '05393837a'
merged_df.loc[(merged_df.eoname == 'Подорожненська середня загальноосвітня школа І-ІІІ ступенів Жидачівського району Львівської області')&(merged_df.eoedrpou=='22390438'), 'eoedrpou'] = '22352780'
merged_df.loc[(merged_df.eoedrpou=='05537242')&(merged_df.KOATUU_2020=='4810137200'), 'eoedrpou'] = '02546186'
merged_df.loc[(merged_df.eoedrpou=='05537242')&(merged_df.KOATUU_2020=='4623010100'), 'eoedrpou'] = '05536975'
merged_df.loc[(merged_df.eoedrpou=='31786185')&(merged_df.KOATUU_2020=='6522955102'), 'eoedrpou'] = '24750622'
merged_df.loc[(merged_df.eoedrpou=='24767373')&(merged_df.KOATUU_2020=='5121481401'), 'eoedrpou'] = '26166956'
merged_df.loc[(merged_df.eoedrpou=='26166956')&(merged_df.KOATUU_2020=='5121285602'), 'eoedrpou'] = '24767373'
merged_df.loc[(merged_df.eoedrpou=='34029703')&(merged_df.KOATUU_2020=='5121480401'), 'eoedrpou'] = '26167051'
merged_df.loc[(merged_df.eoedrpou=='20965633')&(merged_df.KOATUU_2020=='5123155100'), 'eoedrpou'] = '20965633a'
merged_df.loc[(merged_df.eoedrpou=='26455382')&(merged_df.KOATUU_2020=='5123383707'), 'eoedrpou'] = '26455382a'
merged_df.loc[(merged_df.eoedrpou=='26151908')&(merged_df.KOATUU_2020=='6820383001'), 'eoedrpou'] = '21329945'
merged_df.loc[(merged_df.eoedrpou=='33724309')&(merged_df.KOATUU_2020=='5910700000'), 'eoedrpou'] = '33724309a'
merged_df.loc[(merged_df.eoname == 'Державний навчальний заклад "Охтирський центр професійно-технічної освіти" відокремлений навчальний підрозділ м.Тростянець')&(merged_df.eoedrpou=='05537561'), 'eoedrpou'] = '05537561a'
merged_df.loc[(merged_df.eoedrpou=='22593495')&(merged_df.KOATUU_2020=='5925080811'), 'eoedrpou'] = '22593495a'
merged_df.loc[(merged_df.eoedrpou=='24619750')&(merged_df.KOATUU_2020=='6125084101'), 'eoedrpou'] = '37624934'
merged_df.loc[(merged_df.eoedrpou=='25861878')&(merged_df.KOATUU_2020=='6323988501'), 'eoedrpou'] = '25861878a'
merged_df.loc[(merged_df.eoedrpou=='25864569')&(merged_df.KOATUU_2020=='6320681501'), 'eoedrpou'] = '25864569a'
merged_df.loc[(merged_df.eoedrpou=='25792893')&(merged_df.KOATUU_2020=='6321485501'), 'eoedrpou'] = '25792893a'
merged_df.loc[(merged_df.eoedrpou=='24271005')&(merged_df.KOATUU_2020=='6324585001'), 'eoedrpou'] = '24271005a'
merged_df.loc[(merged_df.eoedrpou=='24270046')&(merged_df.KOATUU_2020=='6324855102'), 'eoedrpou'] = '24270046a'
merged_df.loc[(merged_df.eoedrpou=='24958423')&(merged_df.KOATUU_2020=='6525081502'), 'eoedrpou'] = '24958423a'
merged_df.loc[(merged_df.eoedrpou=='25571689')&(merged_df.KOATUU_2020=='6820383001'), 'eoedrpou'] = '21329945'
merged_df.loc[(merged_df.eoedrpou=='25955585')&(merged_df.KOATUU_2020=='7424710150'), 'eoedrpou'] = '25955585a'
merged_df.loc[(merged_df.eoname == 'Коледж економіки і управління Державного вищого навчального закладу "Київський національний економічний університет імені Вадима Гетьмана"')&(merged_df.eoedrpou=='04618613'), 'eoedrpou'] = '00220055'



In [ ]:
merged_df = merged_df.drop_duplicates().reset_index(drop=True)

In [ ]:
merged_df.to_csv('./matching_data/schools_edrpou.csv', index = False)

In [ ]:
merged_df

In [ ]:
merged_df.year = merged_df.year.astype(int)
merged_df_unique = merged_df[['KOATUU_2020', 'KATOTTG_2023', 'eoedrpou', 'year', 'eotypename']]


In [ ]:
merged_df_unique.drop_duplicates(inplace=True)
merged_df_unique[merged_df_unique.duplicated(subset=['year', 'eoedrpou'], keep=False)].eoedrpou.unique()

In [ ]:
merged_df[(merged_df.eoedrpou=='13763076')]


In [ ]:
merged_df_unique.loc[(merged_df_unique.eoedrpou=='26242717')&(merged_df_unique.year == 2023)&(merged_df_unique.eotypename == 'середня загальноосвітня школа'), 'eoedrpou'] = "26242717a"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='05536076')&(merged_df_unique.year == 2018), 'KOATUU_2020'] = "1210136600"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='02541409'), 'KOATUU_2020'] = "1210137800"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='02541846'), 'KOATUU_2020'] = "1224587001"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='02541527'), 'KOATUU_2020'] = "1210400000"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='05536413'), 'KOATUU_2020'] = "1822510100"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='02543880'), 'KOATUU_2020'] = "2310136300"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='26293034'), 'KOATUU_2020'] = "6823355109"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='21295614')&(merged_df_unique.year == 2023), 'KOATUU_2020'] = "4610136900"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='21295614')&(merged_df_unique.year == 2023), 'eotypename'] = "заклад фахової передвищої освіти"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='13763076'), 'KOATUU_2020'] = "3510136600"
merged_df_unique.loc[(merged_df_unique.eoedrpou=='31528905'), 'KOATUU_2020'] = "4820683002"

In [ ]:
merged_df_unique.eotypename.unique()

In [ ]:
merged_df_unique = merged_df_unique.drop_duplicates(subset = ['eoedrpou', 'year']).reset_index(drop=True)

In [ ]:
# load location dictionary for matching names and codes
locations_base = pd.read_csv('./final_tables/locations_base.csv', encoding='utf-8', dtype=str)
locations_base

In [ ]:
merged_df[merged_df.eoedrpou=='31528905']

In [ ]:
set(merged_df_unique.KATOTTG_2023.unique())-set(locations_base.KATOTTG_2023.unique())

In [ ]:
merged_df_unique[merged_df_unique.KATOTTG_2023.isna()]

In [ ]:
merged_df_unique.eotypename.unique()

In [ ]:
# dct_type = {'заклад фахової передвищої освіти':'institution of vocational pre higher education', 
#             'вище професійне училище': 'institution of vocational pre higher education',
#             'заклад професійної (професійно-технічної) освіти':'institution of vocational pre higher education',
#             'вище художнє професійно-технічне училище':'institution of vocational pre higher education',
#             'заклад вищої освіти':'higher education',
#        'Пенітенціарна установа', 'гімназія',
#        'вищий навчальний заклад I-II рівнів акредитації',
#        'спеціальна загальноосвітня школа-інтернат',
#        'центр професійно-технічної освіти',
#        'середня загальноосвітня школа', 'вечірня (змінна) школа', 'ліцей',
#        'спеціалізована школа', 'навчально-виховний комплекс',
#        'спеціальна загальноосвітня школа', 'спеціальна школа',
#        'науковий ліцей', 'професійний ліцей відповідного профілю',
#        'середня загальноосвітня школа-інтернат',
#        'професійно-технічне училище відповідного профілю',
#        'початкова школа', 'колегіум', 'спортивний ліцей',
#        'ліцей із посиленою військово-фізичною підготовкою',
#        'спеціалізована школа-інтернат', 'колегіум/колеж',
#        'навчально-реабілітаційний центр',
#        'загальноосвітня санаторна школа', 'школа соціальної реабілітації',
#        "навчально-виховне об'єднання", 'центр професійної освіти',
#        'центр підготовки і перепідготовки робітничих кадрів', 'коледж',
#        'мистецький ліцей',
#        'військовий (військово-морський, військово-спортивний) ліцей'}

In [ ]:
merged_df_unique.drop(columns=['KOATUU_2020'], inplace=True)

In [ ]:
merged_df_unique[merged_df_unique.duplicated(keep=False)]

In [ ]:
merged_df_unique.to_csv('./final_tables/schools_edrpou.csv', index = False)